# ptsrv.api

> HTTP API for scaled plan and section rendering.

In [ ]:
#| default_exp api

In [ ]:
#| export
"""HTTP API for serving scaled plan / section rasters from E57 scans.

Run:   SCANPLAN_DATA=/path/to/e57s uvicorn scanplan.api:app --host 0.0.0.0 --port 8000

Endpoints (all GET, all return PNG unless ?format=zip|json):

  /scans                          list available scans
  /scans/{name}                   cloud info: extents, floor/ceiling Z, rotation
  /scans/{name}/plan              floor plan       (see query params below)
  /scans/{name}/section           vertical section
  /scans/{name}/elevation         section with large depth, no cut line
  /scans/{name}/reload            drop caches and re-read the E57 (POST)

Common query params
  px        pixel size in metres           default 0.005
  style     color|depth|density|hybrid     default hybrid
  fill      gap-fill radius (m)            default 0.03
  cut_band  thickness of cut slab (m)      default 0.03  (0 = off)
  cut_thicken  dilate the cut line (m)     default 0.0
  bg        background hex colour          default ffffff
  scale     print scale denominator, e.g. 50 -> PNG DPI set for 1:50
  format    png (default) | zip (png+pgw+json) | json (metadata only)

Plan params
  cut       cut height above floor (m)     default 1.2
  level     override floor Z (world)       default detected floor
  below     keep this many m below cut     default down to floor-0.05
  look      down | up (reflected ceiling)  default down
  bounds    xmin,ymin,xmax,ymax crop (world m)

Section params
  a, b      endpoints "x,y" in world metres (required)
  side      left|right  (viewer position walking a->b)  default right
  depth     draw this far beyond the cut plane (m)      default 1.0
  zmin,zmax vertical crop (world Z)

Rendering is CPU-bound and stateless; a single worker on a small box handles a
1M-point cloud at 5 mm/px in well under a second.  Coordinates are in the
ALIGNED frame (see rotation_deg in /scans/{name}); world files are written for
that frame.
"""

SyntaxError: from __future__ imports must occur at the beginning of the file (cloud.py, line 18)

In [ ]:
#| exporti
from __future__ import annotations

import glob
import os
import threading
import time

from fastapi import FastAPI, HTTPException, Query, Response
from fastapi.responses import JSONResponse

from ptsrv.cloud import CloudCache
from ptsrv.render import render_plan, render_section, render_elevation, RenderResult

In [ ]:
#| export
DATA_DIR = os.environ.get("SCANPLAN_DATA", "./data")
MAX_POINTS = int(os.environ.get("SCANPLAN_MAX_POINTS", "0")) or None   # e.g. 20000000 on a small box
ALIGN = os.environ.get("SCANPLAN_ALIGN", "auto")                       # auto | 0 | <degrees>
MAX_PIXELS = int(os.environ.get("SCANPLAN_MAX_PIXELS", "60000000"))
CACHE_ITEMS = int(os.environ.get("SCANPLAN_CACHE_ITEMS", "2"))

app = FastAPI(title="scanplan", version="0.1")
_cache = CloudCache(max_items=CACHE_ITEMS, align=ALIGN if ALIGN == "auto" else float(ALIGN),
                    max_points=MAX_POINTS)
_render_lock = threading.Semaphore(int(os.environ.get("SCANPLAN_CONCURRENCY", "1")))


# ----------------------------------------------------------------------------
def _scan_path(name: str) -> str:
    if "/" in name or "\\" in name or name.startswith("."):
        raise HTTPException(400, "bad scan name")
    for ext in (".e57", ".E57"):
        p = os.path.join(DATA_DIR, name + ext)
        if os.path.isfile(p):
            return p
    raise HTTPException(404, f"scan {name!r} not found in {DATA_DIR}")


def _hex_rgb(s: str) -> tuple[int, int, int]:
    s = s.lstrip("#")
    if len(s) != 6:
        raise HTTPException(400, "colour must be 6 hex digits")
    return tuple(int(s[i:i + 2], 16) for i in (0, 2, 4))


def _pair(s: str, name: str) -> tuple[float, float]:
    try:
        x, y = (float(v) for v in s.split(","))
        return x, y
    except Exception:
        raise HTTPException(400, f"{name} must be 'x,y'")


def _respond(res: RenderResult, fmt: str, basename: str, scale: float | None) -> Response:
    headers = {
        "X-Pixel-Size-M": f"{res.pixel_size:.6f}",
        "X-Origin-U": f"{res.origin[0]:.6f}",
        "X-Origin-V": f"{res.origin[1]:.6f}",
        "X-Points-In-View": str(res.stats.get("points_in_view", 0)),
    }
    if fmt == "png":
        return Response(res.png_bytes(scale), media_type="image/png", headers=headers)
    if fmt == "zip":
        return Response(res.zip_bytes(basename, scale), media_type="application/zip",
                        headers={**headers, "Content-Disposition": f'attachment; filename="{basename}.zip"'})
    if fmt == "json":
        return JSONResponse(res.metadata(), headers=headers)
    raise HTTPException(400, "format must be png, zip or json")


# ----------------------------------------------------------------------------
@app.get("/scans")
def list_scans():
    files = sorted(glob.glob(os.path.join(DATA_DIR, "*.e57")) + glob.glob(os.path.join(DATA_DIR, "*.E57")))
    return [{"name": os.path.splitext(os.path.basename(f))[0],
             "bytes": os.path.getsize(f),
             "cached": os.path.exists(os.path.splitext(f)[0] + ".scanplan.npz")} for f in files]


@app.get("/scans/{name}")
def scan_info(name: str):
    cloud = _cache.get(_scan_path(name))
    d = cloud.info.to_dict()
    d["suggested"] = {
        "plan": {"cut": 1.2, "px": 0.005},
        "section_x": {"a": f"{cloud.info.xmin - 0.2:.2f},{(cloud.info.ymin + cloud.info.ymax) / 2:.2f}",
                      "b": f"{cloud.info.xmax + 0.2:.2f},{(cloud.info.ymin + cloud.info.ymax) / 2:.2f}"},
        "section_y": {"a": f"{(cloud.info.xmin + cloud.info.xmax) / 2:.2f},{cloud.info.ymin - 0.2:.2f}",
                      "b": f"{(cloud.info.xmin + cloud.info.xmax) / 2:.2f},{cloud.info.ymax + 0.2:.2f}"},
    }
    return d


@app.post("/scans/{name}/reload")
def reload_scan(name: str):
    p = _scan_path(name)
    _cache.evict(p)
    cp = os.path.splitext(p)[0] + ".scanplan.npz"
    if os.path.exists(cp):
        os.remove(cp)
    t = time.time()
    _cache.get(p)
    return {"reloaded": name, "seconds": round(time.time() - t, 2)}


@app.get("/scans/{name}/plan")
def plan(name: str,
         cut: float = Query(1.2, description="cut height above floor (m)"),
         level: float | None = None,
         below: float | None = None,
         look: str = Query("down", pattern="^(down|up)$"),
         bounds: str | None = None,
         px: float = Query(0.005, gt=0.0005, le=0.5),
         style: str = Query("hybrid", pattern="^(color|depth|density|hybrid)$"),
         fill: float = 0.03,
         cut_band: float = 0.03,
         cut_thicken: float = 0.0,
         bg: str = "ffffff",
         scale: float | None = None,
         format: str = "png"):
    cloud = _cache.get(_scan_path(name))
    b = None
    if bounds:
        try:
            b = tuple(float(v) for v in bounds.split(","))
            assert len(b) == 4
        except Exception:
            raise HTTPException(400, "bounds must be xmin,ymin,xmax,ymax")
    with _render_lock:
        try:
            res = render_plan(cloud, cut_height=cut, level=level, below=below, look=look,
                              pixel_size=px, style=style, bounds=b, fill_radius=fill,
                              cut_band=cut_band, cut_thicken=cut_thicken, background=_hex_rgb(bg),
                              max_pixels=MAX_PIXELS)
        except ValueError as e:
            raise HTTPException(400, str(e))
    return _respond(res, format, f"{name}_plan_{cut:g}m", scale)


def _section_common(name, a, b, side, depth, zmin, zmax, px, style, fill, cut_band, cut_thicken,
                    bg, scale, format, elevation: bool):
    cloud = _cache.get(_scan_path(name))
    pa, pb = _pair(a, "a"), _pair(b, "b")
    kw = dict(a=pa, b=pb, side=side, zmin=zmin, zmax=zmax, pixel_size=px, style=style,
              fill_radius=fill, cut_thicken=cut_thicken, background=_hex_rgb(bg), max_pixels=MAX_PIXELS)
    with _render_lock:
        try:
            if elevation:
                res = render_elevation(cloud, depth=depth, **kw)
            else:
                res = render_section(cloud, depth=depth, cut_band=cut_band, **kw)
        except ValueError as e:
            raise HTTPException(400, str(e))
    kind = "elev" if elevation else "sect"
    return _respond(res, format, f"{name}_{kind}_{pa[0]:g}_{pa[1]:g}_{pb[0]:g}_{pb[1]:g}", scale)


@app.get("/scans/{name}/section")
def section(name: str, a: str, b: str,
            side: str = Query("right", pattern="^(left|right)$"),
            depth: float = Query(1.0, ge=0.0),
            zmin: float | None = None, zmax: float | None = None,
            px: float = Query(0.005, gt=0.0005, le=0.5),
            style: str = Query("hybrid", pattern="^(color|depth|density|hybrid)$"),
            fill: float = 0.03, cut_band: float = 0.03, cut_thicken: float = 0.0,
            bg: str = "ffffff", scale: float | None = None, format: str = "png"):
    return _section_common(name, a, b, side, depth, zmin, zmax, px, style, fill, cut_band,
                           cut_thicken, bg, scale, format, elevation=False)


@app.get("/scans/{name}/elevation")
def elevation(name: str, a: str, b: str,
              side: str = Query("right", pattern="^(left|right)$"),
              depth: float = Query(50.0, ge=0.0),
              zmin: float | None = None, zmax: float | None = None,
              px: float = Query(0.005, gt=0.0005, le=0.5),
              style: str = Query("hybrid", pattern="^(color|depth|density|hybrid)$"),
              fill: float = 0.03, cut_thicken: float = 0.0,
              bg: str = "ffffff", scale: float | None = None, format: str = "png"):
    return _section_common(name, a, b, side, depth, zmin, zmax, px, style, fill, 0.0,
                           cut_thicken, bg, scale, format, elevation=True)


@app.get("/health")
def health():
    return {"ok": True, "data_dir": DATA_DIR}